# 02 — Features & PD model
Narrative around `src/train.py`: the leakage guard, the out-of-time split, calibration,
the three-way benchmark (HGB vs logistic vs grade-alone), and within-grade re-ranking.
Uses the exact pipeline objects the runner uses.

Prereq: `python run_pipeline.py features` has been run.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
np.seterr(all="ignore")
from sklearn.metrics import roc_auc_score

from src import evaluate, features
from src.config import load
from src.db import connect
from src.train import _fit_calibrated, _grade_benchmark

cfg = load()
con = connect(cfg["paths"]["duckdb"], read_only=True)

## Modelling frame + leakage guard

In [2]:
frame = con.execute("SELECT * FROM v_model_frame").fetchdf()
frame = features.engineer(frame)
features.assert_no_leakage(frame.drop(columns=["default_flag"]))          # raises if a forbidden col slipped in
bench = con.execute("SELECT loan_id, lc_grade FROM mart_loan_benchmark").fetchdf()
frame = frame.merge(bench, on="loan_id", how="left")
frame["split_set"].value_counts()

split_set
test     333721
train    306462
Name: count, dtype: int64

## Out-of-time split (by issue_d, never random)

In [3]:
train = frame[frame.split_set == "train"].reset_index(drop=True)
test = frame[frame.split_set == "test"].reset_index(drop=True)
cols = features.ALLOWLIST + features.DERIVED_NUMERIC
ytr, yte = train.default_flag.to_numpy(), test.default_flag.to_numpy()
print(f"train {len(train):,} (DR {ytr.mean():.3f})   test {len(test):,} (DR {yte.mean():.3f})")
print("features:", cols)

train 306,462 (DR 0.132)   test 333,721 (DR 0.149)
features: ['loan_amnt', 'annual_inc', 'dti', 'emp_length_years', 'fico_mid', 'credit_history_months', 'open_acc', 'total_acc', 'revol_bal', 'revol_util', 'delinq_2yrs', 'inq_last_6mths', 'pub_rec', 'mort_acc', 'acc_open_past_24mths', 'bc_util', 'bc_open_to_buy', 'mo_sin_old_rev_tl_op', 'mths_since_recent_inq', 'mths_since_recent_bc', 'num_actv_bc_tl', 'num_tl_op_past_12m', 'num_accts_ever_120_pd', 'num_tl_90g_dpd_24m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit', 'tot_cur_bal', 'avg_cur_bal', 'tot_coll_amt', 'pub_rec_bankruptcies', 'home_ownership', 'purpose', 'verification_status', 'application_type', 'log_annual_inc', 'loan_to_income']


## Fit the primary model + inspect calibration

In [4]:
model = _fit_calibrated(cfg["model"]["type"], train[cols], ytr, cfg)
p_te = model.predict_proba(test[cols])[:, 1]
evaluate.summary(yte, p_te)

{'auc': 0.6908074736870696,
 'gini': 0.38161494737413926,
 'ks': 0.2763061715702882,
 'brier': 0.11931714289046118,
 'base_rate': 0.14878895844133272,
 'n': 333721}

In [5]:
fig_dir = cfg["paths"]["figures_dir"]
evaluate.plot_calibration(yte, p_te, f"{fig_dir}/nb_calibration.png", "Calibration — out-of-time test")
dec = evaluate.decile_table(yte, p_te)
dec.round(4)

,bucket,n,mean_pd,obs_rate,lift
0,0,33373,0.0314,0.0314,0.2111
1,1,33372,0.0558,0.0591,0.3974
2,2,33372,0.0749,0.0813,0.5462
3,3,33372,0.0932,0.1036,0.6966
4,4,33372,0.1129,0.1236,0.8305
5,5,33372,0.1321,0.1463,0.9834
6,6,33371,0.1543,0.1720,1.1562
7,7,33373,0.1837,0.2032,1.3656
8,8,33372,0.2251,0.2406,1.6170
9,9,33372,0.2990,0.3267,2.1960


## Three-way comparison: primary vs the other model type vs grade-alone

In [6]:
prim = cfg["model"]["type"]
sec = "logistic" if prim == "hgb" else "hgb"
sec_model = _fit_calibrated(sec, train[cols], ytr, cfg)
p_sec = sec_model.predict_proba(test[cols])[:, 1]
grade_bm = _grade_benchmark(train[["lc_grade"]], test[["lc_grade"]], ytr, yte)
pd.Series({
    f"{prim} (primary)": roc_auc_score(yte, p_te),
    f"{sec}": roc_auc_score(yte, p_sec),
    "grade-only": grade_bm["auc"],
}).round(4)

hgb (primary)    0.6908
logistic         0.6803
grade-only       0.6688
dtype: float64

## Does the model re-rank *within* a grade?
Observed default rate by grade x model-PD-decile. If the model adds nothing over grade,
rows are flat.

In [7]:
t = test.copy()
t["pd_hat"] = p_te
t["pd_decile"] = pd.qcut(t.pd_hat.rank(method="first"), 10, labels=False)
(t.groupby(["lc_grade", "pd_decile"]).default_flag.mean().unstack().round(3))

pd_decile,0,1,2,3,4,5,6,7,8,9
lc_grade,,,,,,,,,,
A,0.027,0.047,0.060,0.074,0.082,0.097,0.113,0.131,0.127,0.203
B,0.050,0.066,0.083,0.099,0.115,0.130,0.141,0.168,0.189,0.244
C,0.079,0.105,0.118,0.134,0.150,0.165,0.186,0.207,0.233,0.293
D,0.113,0.137,0.165,0.168,0.167,0.201,0.234,0.243,0.285,0.343
E,0.250,0.100,0.172,0.208,0.256,0.213,0.255,0.289,0.313,0.397
F,NaN,NaN,0.400,0.000,0.241,0.333,0.276,0.377,0.362,0.469
G,NaN,NaN,NaN,1.000,0.000,0.000,0.222,0.435,0.405,0.488


## Calibration by grade — where does PD miss?

In [8]:
(t.groupby("lc_grade")
   .agg(n=("loan_id", "size"), pred_pd=("pd_hat", "mean"), obs_dr=("default_flag", "mean"))
   .round(3))

,n,pred_pd,obs_dr
lc_grade,,,
A,82786,0.068,0.054
B,110329,0.122,0.119
C,90335,0.171,0.195
D,37241,0.208,0.265
E,10994,0.243,0.333
F,1712,0.275,0.423
G,324,0.311,0.460


In [9]:
con.close()